# Week 06 — Leakage-aware validation and model interpretation

**Seismic Python & AI Engineering | Hands-on notebook**

Model scores are meaningful only if the validation design matches how the model will be used. Correlated seismic observations make leakage a central risk.

**Estimated independent study:** 4–5 hours in this notebook, plus 2–3 hours on repository exercises and review.  
**Data:** entirely synthetic; no external API, proprietary dataset, or GPU required.  
**Prerequisite:** complete the root README setup and open Jupyter from the repository root.

### Learning outcomes
- Explain sample independence and grouped validation.
- Use GroupKFold and compare random vs group splits.
- Analyze performance variability, confusion and feature importance.
- Write a reproducible validation report with explicit limitations.

### How to study
Read the explanations, execute the worked cells in order, complete the **Your turn** tasks, and implement the week's function in `src/seismic_toolkit/challenges.py`. Run the targeted exercise test to assess progress. The initial baseline tests should pass even before you complete the challenges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from seismic_toolkit import __version__
print(f"Seismic toolkit version: {__version__}")
np.set_printoptions(precision=4, suppress=True)

## 1. Why a random split can be misleading
Adjacent traces, overlapping windows, time-lapse vintages and neighboring wells may share information. A random row split mixes highly similar samples across train and test. If deployment is to new survey areas or new wells, validation should separate those physical groups.

Choosing the proper group is a domain-specific decision: group by well, field, survey tile, acquisition vintage or another entity matching the real generalization goal. There is no universal split rule.

In [ ]:
from seismic_toolkit.datasets import FEATURE_COLUMNS, make_synthetic_facies_dataset
from seismic_toolkit.ml import make_baseline_model, grouped_cv_accuracy
from sklearn.model_selection import cross_val_score, StratifiedKFold

frame = make_synthetic_facies_dataset(n_groups=16, samples_per_group=35, seed=2026)
X = frame[list(FEATURE_COLUMNS)]
y = frame["facies"]
naive = cross_val_score(make_baseline_model(), X, y, cv=StratifiedKFold(4, shuffle=True, random_state=1), scoring="accuracy")
by_group = grouped_cv_accuracy(frame, n_splits=4)
print("Random-row CV:", np.round(naive, 3), "mean:", round(float(naive.mean()), 3))
print("Grouped CV:   ", np.round(by_group, 3), "mean:", round(float(by_group.mean()), 3))

## 2. Interpreting the comparison responsibly
On simple toy data, grouped scores do not necessarily come out lower; the point is **methodological validity**, not forcing a preconceived gap. Each split estimates performance for a different deployment scenario.

**Your turn A:** state which validation method is more defensible when predicting a new well and which might be acceptable when interpolating independent records from the same distribution. Name two remaining risks even after grouping.

## 3. GroupKFold and split hygiene
`GroupKFold` keeps every group's rows together and ensures groups do not appear in both training and validation within the same fold. Putting `SimpleImputer` and `StandardScaler` inside the `Pipeline` prevents preprocessing leakage during each fold.

In [ ]:
from sklearn.model_selection import GroupKFold
for fold, (train_idx, val_idx) in enumerate(GroupKFold(n_splits=4).split(X, y, groups=frame.group), 1):
    train_groups = set(frame.group.iloc[train_idx])
    val_groups = set(frame.group.iloc[val_idx])
    assert train_groups.isdisjoint(val_groups)
    print(f"Fold {fold}: {len(train_idx)} train, {len(val_idx)} validation, {len(val_groups)} validation groups")

## 4. Feature importance is not a geological explanation
Permutation importance quantifies how much a fitted model's performance changes when a feature is shuffled in a particular evaluation set. It does not establish causality, and correlated variables can distribute importance in misleading ways.

**Your turn B:** generate permutation importance on group-held-out data. Repeat with a different random seed and compare ranking stability.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.inspection import permutation_importance
train_idx, val_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7).split(X, y, frame.group))
model = make_baseline_model().fit(X.iloc[train_idx], y.iloc[train_idx])
imp = permutation_importance(model, X.iloc[val_idx], y.iloc[val_idx], scoring="accuracy", n_repeats=5, random_state=7)
for feature, mean, std in zip(FEATURE_COLUMNS, imp.importances_mean, imp.importances_std):
    print(f"{feature:10}: mean delta={mean:+.3f} ±{std:.3f}")

## 5. Independent implementation — Week 6 challenge
Implement `evaluate_by_group(frame, n_splits=4)` in `challenges.py`.

Return one accuracy per fold as a NumPy array of shape `(n_splits,)`. Respect group separation. Refit all steps in each fold. Reject invalid `n_splits`.

**Applied problem:** describe a validation scheme for repeated 4D surveys when baseline and monitor from the same physical location are strongly correlated. Specify *what generalization claim* your score supports and which claim it does not.

In [ ]:
print("Group CV mean±std:", float(np.mean(by_group)), float(np.std(by_group)))
assert by_group.shape == (4,)
assert np.all((0 <= by_group) & (by_group <= 1))

## 6. Report template
A short validation report should state: (a) prediction task and units; (b) data provenance, sample size and group definition; (c) split strategy; (d) preprocessing **inside** folds; (e) metrics and dispersion; (f) most important failure patterns; (g) deployment limitations.

**Stretch:** quantify uncertainty by repeating grouped shuffle splits with several seeds. Do not report a narrow interval as a guarantee of performance on unknown geological settings.

---

## Weekly submission and self-review

1. Complete the independent exercise in `src/seismic_toolkit/challenges.py`.
2. Run: `python -m pytest -o addopts='' -m exercise -k week6 -q`.
3. Run baseline regression tests: `python -m pytest -q`.
4. Commit a focused change to a feature branch; document what changed and why.
5. Record one failure mode, one design decision, and one next-step question in your learning journal.

**Further work:** see [`../exercises/week06.md`](../exercises/week06.md) for a graded problem, extra practice and acceptance criteria. Compare with `solutions/reference_challenges.py` only **after** attempting your own solution.

**Important:** all datasets and algorithms are educational demonstrations, not validated reservoir-interpretation systems.